# Assignment 2 GPU Training on Google Colab

This notebook trains the `Jorryt` branch on a Colab GPU, evaluates frozen encoder embeddings, and saves checkpoints back to Google Drive.

Before running: in Colab, go to **Runtime > Change runtime type > Hardware accelerator > GPU**.

## Put Your Data In Drive

Recommended Drive layout:

```text
MyDrive/
  assignment_2_data/
    data/
      Intra/
        train/
          *.h5
        test/
          *.h5
      Cross/
        train/
          *.h5
        test/
          *.h5
```

If you downloaded a zip, extract the **contents** of the zip into `MyDrive/assignment_2_data/data/`, so that `Intra` and `Cross` are directly inside the `data` folder.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No CUDA GPU is available. In Colab, enable Runtime > Change runtime type > GPU.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/assignment_2_data/data")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/assignment_2_outputs")

required_dirs = [
    DRIVE_DATA_DIR / "Intra" / "train",
    DRIVE_DATA_DIR / "Intra" / "test",
]

for directory in required_dirs:
    if not directory.exists():
        raise FileNotFoundError(f"Missing expected data directory: {directory}")

train_files = list((DRIVE_DATA_DIR / "Intra" / "train").glob("*.h5"))
test_files = list((DRIVE_DATA_DIR / "Intra" / "test").glob("*.h5"))
print("Train files:", len(train_files))
print("Test files:", len(test_files))
if not train_files:
    raise FileNotFoundError(f"No .h5 files found in {DRIVE_DATA_DIR / 'Intra' / 'train'}")
if not test_files:
    raise FileNotFoundError(f"No .h5 files found in {DRIVE_DATA_DIR / 'Intra' / 'test'}")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

repo_dir = Path("/content/assignment-2")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

repo_url = os.environ.get("GITHUB_REPO_URL", "https://github.com/jorrytdejong/assignment-2.git")
github_token = os.environ.get("GITHUB_TOKEN")

if github_token:
    repo_url = repo_url.replace("https://", f"https://{github_token}@")

result = subprocess.run(["git", "clone", repo_url, str(repo_dir)], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "Git clone failed. If the repo is private, set GITHUB_TOKEN in Colab, or make the repo public before running this cell."
    )

result = subprocess.run(["git", "checkout", "Jorryt"], cwd=str(repo_dir), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Failed to checkout the Jorryt branch.")

print(f"Cloned repo into {repo_dir}")

In [ ]:
from pathlib import Path

repo_data = Path("/content/assignment-2/data")
if repo_data.exists() or repo_data.is_symlink():
    !rm -rf /content/assignment-2/data

!ln -s "{DRIVE_DATA_DIR}" /content/assignment-2/data
!find /content/assignment-2/data/Intra -maxdepth 2 -type f -name '*.h5' | head

In [ ]:
%cd /content/assignment-2
!pwd
!pip install -q uv
!uv sync

## Configure Model Runs

Each entry creates a separate output folder under `runs/`, so checkpoints and embedding results do not overwrite each other. Start with `cnn`; add `lstm` once the full pipeline works.


In [ ]:
MODEL_RUNS = [
    {
        "name": "cnn",
        "model_type": "cnn",
        "max_epochs": 10,
        "batch_size": 8,
        "window_size": 2048,
        "window_stride": 2048,
    },
    {
        "name": "tcn",
        "model_type": "tcn",
        "max_epochs": 10,
        "batch_size": 8,
        "window_size": 2048,
        "window_stride": 2048,
        "tcn_channels": 128,
        "tcn_num_blocks": 5,
    },
    {
        "name": "transformer",
        "model_type": "transformer",
        "max_epochs": 10,
        "batch_size": 4,
        "window_size": 512,
        "window_stride": 512,
        "transformer_d_model": 128,
        "transformer_num_heads": 4,
        "transformer_num_layers": 2,
    },
    {
        "name": "cnn2d",
        "model_type": "cnn2d",
        "max_epochs": 10,
        "batch_size": 8,
        "window_size": 1024,
        "window_stride": 1024,
        "cnn2d_base_channels": 32,
        "cnn2d_num_blocks": 3,
    },
    {
        "name": "lstm",
        "model_type": "lstm",
        "max_epochs": 10,
        "batch_size": 8,
        "window_size": 512,
        "window_stride": 512,
        "lstm_hidden_dim": 128,
        "lstm_num_layers": 1,
    },
]

for run in MODEL_RUNS:
    run["checkpoint_dir"] = f"runs/{run['name']}/checkpoints"
    run["log_dir"] = f"runs/{run['name']}/logs"
    run["results_csv"] = f"runs/{run['name']}/embedding_results.csv"

MODEL_RUNS


In [ ]:
import subprocess

OPTION_KEYS = [
    "lstm_hidden_dim", "lstm_num_layers", "lstm_dropout",
    "tcn_channels", "tcn_num_blocks", "tcn_kernel_size", "tcn_dropout",
    "transformer_d_model", "transformer_num_heads", "transformer_num_layers",
    "transformer_dim_feedforward", "transformer_dropout",
    "cnn2d_base_channels", "cnn2d_num_blocks",
]

for run in MODEL_RUNS:
    print(f"\n=== Training {run['name']} ===")
    command = [
        "uv", "run", "train_model.py",
        "--model-type", run["model_type"],
        "--max-epochs", str(run["max_epochs"]),
        "--batch-size", str(run["batch_size"]),
        "--window-size", str(run["window_size"]),
        "--window-stride", str(run["window_stride"]),
        "--accelerator", "gpu",
        "--checkpoint-dir", run["checkpoint_dir"],
        "--log-dir", run["log_dir"],
    ]
    for key in OPTION_KEYS:
        if key in run:
            command.extend([f"--{key.replace('_', '-')}", str(run[key])])
    subprocess.run(command, check=True)


In [ ]:
import subprocess

for run in MODEL_RUNS:
    print(f"\n=== Evaluating {run['name']} ===")
    subprocess.run([
        "uv", "run", "evaluate_embeddings.py",
        "--model-type", run["model_type"],
        "--checkpoint-dir", run["checkpoint_dir"],
        "--all-checkpoints",
        "--batch-size", "32",
        "--num-workers", "2",
        "--device", "cuda",
        "--output-csv", run["results_csv"],
    ], check=True)


In [ ]:
!mkdir -p "{DRIVE_OUTPUT_DIR}"
!cp -r runs "{DRIVE_OUTPUT_DIR}/runs"
!find "{DRIVE_OUTPUT_DIR}/runs" -maxdepth 4 -type f | sort | tail -40


## Optional: Faster/Slower Configs

For faster iteration, reduce `max_epochs` to 2 or comment out model entries in `MODEL_RUNS`.

TCN is the closest comparison to the current CNN and can usually use the same `window_size = 2048`.

Transformer and LSTM should start smaller (`window_size = 512`) because they are much more expensive over long sequences.

2D CNN is included as an empirical baseline, but interpret it carefully unless the 248 sensor order has meaningful spatial adjacency.
